In [1]:
import scanpy as sc
import muon as mu

/home/miniconda3/envs/citeseq/lib/python3.12/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
def preprocess_rna(file_path):
    # 1. Data loading (AnnData)
    print("Data loading...")
    mdata = mu.read(file_path) # [cite: 192, 193]
    adata = mdata.mod['rna']
    
    # 2. Filtering (too small cells & genes)
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    
    # 3. Copying data (Preserve raw count)
    adata.layers["counts"] = adata.X.copy()
    
    # 4. Normalization & log transfer 
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    # 5. Select HVG (Highly Variable Genes)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)
    
    print(f"Preprocessing done: {adata.n_obs} cells, {adata.n_vars} genes Selected")
    return adata

In [3]:
file_path = 'GSE194315_binarized_mu.h5mu'
adata_preprocessed = preprocess_rna(file_path)
adata_preprocessed

Data loading...


/home/miniconda3/envs/citeseq/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/miniconda3/envs/citeseq/lib/python3.12/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Preprocessing done: 180794 cells, 2000 genes Selected


AnnData object with n_obs × n_vars = 180794 × 2000
    obs: 'sample', 'batch_ID', 'Sample', 'Run', 'Subject', 'Status', 'DemuxletDropletType', 'IncludedInStudy', 'CellType', 'Cluster', 'n_genes'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    layers: 'counts'

In [4]:
!ls -lh GSE194315_binarized_mu.h5mu

-rw------- 1 ch-epfl-423059 ubuntu 2.3G Apr 16 10:23 GSE194315_binarized_mu.h5mu


In [4]:
import torch
import numpy as np
from helical.models.scgpt import scGPT, scGPTConfig

2026-04-16 19:59:56,777 - INFO:datasets:PyTorch version 2.7.0 available.


In [5]:
def extract_and_cache_rna(adata):
    print("Extracting RNA encodings using GPU...")
    
    if "counts" in adata.layers:
        print("Using raw counts from adata.layers['counts']...")
        adata.X = adata.layers["counts"].copy()
    else:
        print("Warning: 'counts' layer not found. Attempting to use adata.X as is.")

    import numpy as np
    from scipy.sparse import issparse
    
    if issparse(adata.X):
        adata.X.data = np.round(adata.X.data).astype(int)
    else:
        adata.X = np.round(adata.X).astype(int)

    config = scGPTConfig()
    config.config["device"] = "cuda"
    model = scGPT(configurer=config)
    
    try:
        print("Processing data for scGPT (Tokenizing integers)...")
        processed_data = model.process_data(adata)
        
        print("Getting embeddings on GPU ...")
        rna_encodings = model.get_embeddings(processed_data)
        
        if hasattr(rna_encodings, "detach"):
            rna_encodings = rna_encodings.detach().cpu().numpy()
            
        np.save('rna_encodings.npy', rna_encodings)
        print(f"Successed! {rna_encodings.shape}")
        return rna_encodings

    except Exception as e:
        print(f"Error: {e}")
        return None

rna_vectors = extract_and_cache_rna(adata_preprocessed)

Extracting RNA encodings using GPU...
Using raw counts from adata.layers['counts']...


2026-04-16 20:00:26,600 - WARNING:py.warnings:/home/miniconda3/envs/citeseq/lib/python3.12/site-packages/helical/models/scgpt/model_dir/transformer.py:135: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  warnings.warn(

2026-04-16 20:00:43,353 - INFO:helical.models.scgpt.model:Model finished initializing.
2026-04-16 20:00:43,355 - INFO:helical.models.scgpt.model:'scGPT' model is in 'eval' mode, on device 'cuda' with embedding mode 'cell'.
2026-04-16 20:00:43,358 - INFO:helical.models.scgpt.model:Processing data for scGPT.
2026-04-16 20:00:43,395 - INFO:helical.models.scgpt.model:Filtering out 149 genes to a total of 1851 genes with an ID in the scGPT vocabulary.


Processing data for scGPT (Tokenizing integers)...


2026-04-16 20:00:43,655 - INFO:helical.models.scgpt.model:Successfully processed the data for scGPT.
2026-04-16 20:00:43,656 - INFO:helical.models.scgpt.model:Started getting embeddings:


Getting embeddings on GPU ...


Embedding cells: 100%|██████████| 7534/7534 [03:06<00:00, 40.38it/s]
2026-04-16 20:03:51,141 - INFO:helical.models.scgpt.model:Finished getting embeddings.


Successed! (180794, 512)


In [6]:
print(rna_vectors[0, :10]) 
print(rna_vectors.shape)

[ 0.02228101 -0.01261382  0.00673866  0.01851242 -0.00213987 -0.00930705
 -0.00593815 -0.00154665 -0.03349824  0.04318295]
(180794, 512)


In [8]:
import numpy as np

if rna_vectors.shape[0] == 512:
    rna_vectors = rna_vectors.T

print(f"Final vector: {rna_vectors.shape}")

if np.isnan(rna_vectors).any():
    print("Warning: NaN included.")
else:
    adata_preprocessed.obsm['X_scGPT_harmony'] = rna_vectors
    np.save('final_rna_vectors.npy', rna_vectors)

Final vector: (180794, 512)


In [9]:
print(f"현재 데이터 세포 수: {adata_preprocessed.n_obs}")
print(f"추출된 벡터 크기: {rna_vectors.shape}")

현재 데이터 세포 수: 180794
추출된 벡터 크기: (180794, 512)
